# 🛡️ In-the-Wild AI Face Detection — Fast 10-Minute Training
### Training on 140k Real & Fake Faces (Flickr FFHQ vs. StyleGAN / ThisPersonDoesNotExist)

This lightweight notebook trains **DINOv2** and **EfficientNet-B0** on real camera portraits vs. modern AI-generated faces.  
Because faces are already cropped, there is **zero MTCNN face extraction needed** — the whole notebook finishes in ~10 minutes on Kaggle GPU!

In [ ]:
# ── Cell 1: GPU Diagnostics ──────────────────────────────────────
import subprocess, torch

print("=" * 60)
print("  GPU Diagnostics")
print("=" * 60)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device     :", torch.cuda.get_device_name(0))
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Please enable GPU accelerator in Settings (top-right)")


In [ ]:
%%time
# ── Cell 2: Dependencies & Repository Setup ───────────────────────
!pip install -q timm albumentations grad-cam tabulate seaborn pyyaml

import os
REPO_URL = "https://github.com/mamunur77/Deepfake.git"
PROJECT_DIR = "/kaggle/working/deepfake-vit"

if os.path.exists(PROJECT_DIR):
    print(f"[setup] Updating repository at {PROJECT_DIR}...")
    !cd {PROJECT_DIR} && git pull origin main
else:
    print(f"[setup] Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"[setup] Working directory: {os.getcwd()}")


In [ ]:
%%time
# ── Cell 3: Build Fast Balanced Splits (No Face Extraction Needed) ─
import os, glob
import pandas as pd

# Locate 140k dataset root folder
candidates = glob.glob("/kaggle/input/**/real_vs_fake", recursive=True) + glob.glob("/kaggle/input/**/real-vs-fake", recursive=True)
data_root = candidates[0] if candidates else "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"
print(f"[setup] Dataset detected at: {data_root}")

os.makedirs("data/splits", exist_ok=True)

# Sample a balanced subset for rapid, high-accuracy training
for split_name, n_per_class in [("train", 5000), ("valid", 1000), ("test", 1000)]:
    split_dir = os.path.join(data_root, split_name)
    rows = []
    for label_str, label_idx in [("real", 0), ("fake", 1)]:
        folder = os.path.join(split_dir, label_str)
        files = sorted(glob.glob(os.path.join(folder, "*.jpg")))[:n_per_class]
        for f in files:
            vid_id = os.path.splitext(os.path.basename(f))[0]
            rows.append({"frame_path": f, "video_id": vid_id, "label": label_idx, "label_name": label_str})
    
    out_csv = f"data/splits/{'val' if split_name == 'valid' else split_name}.csv"
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"  {out_csv}: {len(df):,} images (Real: {(df['label']==0).sum():,}, Fake: {(df['label']==1).sum():,})")

print("\n[splits] Balanced train/val/test splits generated successfully!")


In [ ]:
%%time
# ── Cell 4: Train DINOv2 & EfficientNet-B0 (5 Epochs, Batch 64) ───
import os, yaml

configs = ["configs/dinov2_partial.yaml", "configs/efficientnet_b0.yaml"]

# Apply fast hyperparameters
for cfg_path in configs:
    with open(cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['epochs'] = 5
    cfg['training']['batch_size'] = 64
    cfg['training']['early_stopping_patience'] = 3
    with open(cfg_path, 'w') as f:
        yaml.safe_dump(cfg, f)

for cfg_path in configs:
    print("\n" + "=" * 70)
    print(f"  Training: {cfg_path}")
    print("=" * 70)
    exit_code = os.system(f"python scripts/train.py --config {cfg_path}")
    if exit_code != 0:
        print(f"  [WARNING] Training failed for {cfg_path}")


In [ ]:
%%time
# ── Cell 5: Evaluation & Metric Reporting ─────────────────────────
!python scripts/evaluate.py --checkpoint outputs/dinov2_partial/best_model.pt --split-csv data/splits/test.csv
!python scripts/evaluate.py --checkpoint outputs/efficientnet_b0/best_model.pt --split-csv data/splits/test.csv
!python scripts/compare_runs.py


In [ ]:
# ── Cell 6: Zip Outputs & Generate Clickable Download Link ────────
import os
from IPython.display import FileLink, HTML

!zip -r /kaggle/working/deepfake_ai_faces.zip outputs/ -x "*.tfevents*"

os.chdir('/kaggle/working')
print("\n" + "=" * 60)
print("  Click below to download your trained AI-Face models:")
print("=" * 60)
display(FileLink('deepfake_ai_faces.zip'))
